# CBraMod · PhysioNet MI (4 classes) — A100 accelerated reproduction

A fresh Colab A100 notebook based on the supplied **working T4 notebook**. The only file required in Drive before running is the original PhysioNet ZIP in **`MyDrive/CBraMod-MI/`**. All derived folders are created automatically.

**Main goal: reduce end-to-end runtime while keeping the experiment recognizable.** By default, A100 **fast mode** uses BF16 autocast, TF32 matmuls, RAM-cached EEG samples, pinned-memory loaders and avoids per-step GPU synchronizations. The **same** 109 subjects, 70/19/20 split, 9,837 trials, CBraMod checkpoint, trainable backbone, batch size 64, seed 3407, optimizer settings and 50 epochs are retained. BF16 changes floating-point arithmetic, so results will not match the T4 run bit-for-bit.

For the closest numerical comparison, set `FAST_A100 = False` in Cell 2. This disables BF16/TF32 and RAM caching while using the original training settings. **Changing batch size is not necessary to obtain A100 speedups** and would change the effective training schedule and classifier learning rate.

**New standard output location:** `MyDrive/CBraMod-MI/results/physionet_mi_4class/<experiment>/`, with `runs/`, `checkpoints/`, `logs/`, and `metrics/`.

Select **Runtime → Change runtime type → A100 GPU** and run all cells in order. The personal GitHub notebook/repo workflow is not managed here; the official CBraMod implementation remains a training dependency, pinned to the commit used in the previous notebook.

## 1 · Drive, experiment options and organized paths

Change only the values marked **user settings**. All training and preprocessing happens on `/content` (faster local storage). Only **finished run artifacts** are copied to Drive. Dataset ZIP is staged on local disk once so EDF extraction avoids repeated Drive reads.

In [ ]:
from pathlib import Path, PurePosixPath
from google.colab import drive
import shutil, subprocess, sys, re, os, json, hashlib, time

# ============= USER SETTINGS =============
FAST_A100 = True           # True = faster A100 BF16 training; False = original FP32 arithmetic
SEEDS = [3407]
EPOCHS = 50
BATCH_SIZE = 64            # Preserve the T4 training configuration
NUM_WORKERS = 2 if FAST_A100 else 0
FORCE_RERUN = False        # False = keep completed results already saved on Drive
EXPERIMENT_SUFFIX = ''     # Optional distinct tag, e.g. 'replicate_02'
# =========================================

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

DRIVE_FOLDER = Path('/content/drive/MyDrive/CBraMod-MI')
ZIP_PATH = DRIVE_FOLDER / 'eeg-motor-movementimagery-dataset-1.0.0.zip'
if not ZIP_PATH.is_file():
    candidates = sorted(DRIVE_FOLDER.glob('*.zip')) if DRIVE_FOLDER.is_dir() else []
    if len(candidates) == 1:
        ZIP_PATH = candidates[0]
        print('Found dataset ZIP:', ZIP_PATH.name)
    else:
        raise FileNotFoundError(f'Expected dataset ZIP in {DRIVE_FOLDER}; found: {candidates}')

MODE = 'a100_fast_bf16' if FAST_A100 else 'fp32_reference'
EXPERIMENT_ID = f'{MODE}_bs{BATCH_SIZE}_e{EPOCHS}'
if EXPERIMENT_SUFFIX:
    if not re.fullmatch(r'[A-Za-z0-9_-]+', EXPERIMENT_SUFFIX):
        raise ValueError('EXPERIMENT_SUFFIX may contain only letters, digits, - and _.')
    EXPERIMENT_ID += '_' + EXPERIMENT_SUFFIX

WORK = Path('/content/cbramod_experiment')
LOCAL_ZIP = WORK / 'downloads' / ZIP_PATH.name
RAW = WORK / 'raw'
DB_PATH = WORK / 'physionet_lmdb'
CBRAMOD = Path('/content/CBraMod')
RESULTS_ROOT = DRIVE_FOLDER / 'results'
DRIVE_EXPERIMENT = RESULTS_ROOT / 'physionet_mi_4class' / EXPERIMENT_ID
LOCAL_EXPERIMENT = WORK / 'experiments' / EXPERIMENT_ID
RUNS = LOCAL_EXPERIMENT / 'runs'
REPRO_COMMIT = 'b9e961003214326972c567eff390e75b0287e32a'

for folder in [LOCAL_ZIP.parent, RAW, DB_PATH.parent, RUNS]:
    folder.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage('/content').free / 1e9
print('Source:', ZIP_PATH, '| size GB:', round(ZIP_PATH.stat().st_size / 1e9, 2))
print('Free local disk GB:', round(free_gb, 1))
if free_gb < 12:
    raise RuntimeError('Need at least 12 GB free on /content for staging, EDF and LMDB. Free local disk.')
if not LOCAL_ZIP.is_file() or LOCAL_ZIP.stat().st_size != ZIP_PATH.stat().st_size:
    print('Staging dataset ZIP from Drive to local Colab disk...')
    shutil.copy2(ZIP_PATH, LOCAL_ZIP)
print('Local dataset ZIP:', LOCAL_ZIP)
print('Training mode:', MODE, '| workers:', NUM_WORKERS)
print('Drive results:', DRIVE_EXPERIMENT)

## 2 · CBraMod source and Python dependencies

Uses the official source pinned to the same upstream commit as the previous working notebook, **not** your personal GitHub workflow. All dataset and experiment artifacts are written outside the source tree. Packages and GPU environment are recorded with the experiment.

In [ ]:
# Fetch/check out only the official upstream source; ignore personal GitHub workflow.
if not (CBRAMOD / '.git').is_dir():
    if CBRAMOD.exists() and any(CBRAMOD.iterdir()):
        raise RuntimeError(f'{CBRAMOD} exists but is not a Git checkout. Choose an empty path.')
    subprocess.run(['git', 'clone', 'https://github.com/wjq-learning/CBraMod.git', str(CBRAMOD)], check=True)

commit_before = subprocess.check_output(
    ['git', '-C', str(CBRAMOD), 'rev-parse', 'HEAD'], text=True
).strip()
if commit_before != REPRO_COMMIT:
    subprocess.run(['git', '-C', str(CBRAMOD), 'fetch', 'origin', REPRO_COMMIT], check=True)
    subprocess.run(['git', '-C', str(CBRAMOD), 'checkout', '--detach', REPRO_COMMIT], check=True)

commit = subprocess.check_output(
    ['git', '-C', str(CBRAMOD), 'rev-parse', 'HEAD'], text=True
).strip()
assert commit == REPRO_COMMIT, (commit, REPRO_COMMIT)
assert (CBRAMOD / 'finetune_main.py').is_file()

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(CBRAMOD / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
print('Official CBraMod Git commit:', commit)

In [ ]:
# Check A100 and record the main software versions.
import torch, numpy as np, mne, lmdb
import importlib.metadata as im

assert torch.cuda.is_available(), 'No CUDA GPU detected. Select an A100 GPU runtime in Colab.'
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} ({gpu_mem_gb:.1f} GiB VRAM)')
if FAST_A100 and not torch.cuda.is_bf16_supported():
    raise RuntimeError('Fast BF16 mode requires a BF16-capable GPU (select A100).')
if 'A100' not in gpu_name.upper():
    print('NOTE: This is not an A100. Training can still run, but memory/speed may differ.')

VERSIONS = {
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'mne': mne.__version__,
    'numpy': np.__version__,
    'lmdb': im.version('lmdb'),
    'scipy': im.version('scipy'),
    'scikit-learn': im.version('scikit-learn'),
    'huggingface_hub': im.version('huggingface_hub'),
}
print(json.dumps(VERSIONS, indent=2))

## 3 · Extract the six motor-*imagery* runs (654 EDFs)

Runs 04, 06, 08, 10, 12, and 14 from 109 subjects are read from the locally staged ZIP. Extraction is skipped when a matching-size file already exists. No Drive folder is used for extracted EEG files.

In [ ]:
# Find the 6 imagery EDF runs for all 109 subjects
from zipfile import ZipFile
from pathlib import PurePosixPath
import re

runs = ['04', '06', '08', '10', '12', '14']
files_to_extract = []

with ZipFile(LOCAL_ZIP) as z:
    for info in z.infolist():
        filename = PurePosixPath(info.filename).name
        if re.fullmatch(r'S\d{3}R(04|06|08|10|12|14)\.edf', filename, re.I):
            filename = filename[:7].upper() + '.edf'
            files_to_extract.append((info.filename, filename, info.file_size))

print('Imagery EDF files:', len(files_to_extract), '/ 654')
print('First few:', files_to_extract[:3])
assert len(files_to_extract) == 654, 'The ZIP should contain 654 imagery EDFs.'


In [ ]:
# Extract the files and fix uppercase .EDF filenames from the old notebook
RAW.mkdir(parents=True, exist_ok=True)
renamed = 0
copied = 0

with ZipFile(LOCAL_ZIP) as z:
    for archive_name, filename, size in files_to_extract:
        subject = filename[:4]
        folder = RAW / subject
        folder.mkdir(parents=True, exist_ok=True)
        dest = folder / filename

        old_name = folder / filename.upper()
        if old_name.exists() and not dest.exists() and old_name.stat().st_size == size:
            old_name.rename(dest)
            renamed += 1

        if dest.exists() and dest.stat().st_size == size:
            continue

        with z.open(archive_name) as source, dest.open('wb') as target:
            shutil.copyfileobj(source, target)
        copied += 1

extracted = list(RAW.glob('S*/*.edf'))
print('Fixed uppercase filenames:', renamed)
print('New files extracted:', copied)
print('Files ready:', len(extracted), '/ 654')
print('Example exists:', (RAW / 'S001' / 'S001R04.edf').exists())
assert len(extracted) == 654, 'Extraction is incomplete. Check the paths printed above.'


## 4 · Annotation-to-class mapping

The EDF annotation codes `T0`, `T1`, `T2` are interpreted according to the run number. **Discard rest events (`T0`).**

| Runs | `T1` | `T2` |
|:--|:--|:--|
| 04 / 08 / 12 | 0 · imagined left fist | 1 · imagined right fist |
| 06 / 10 / 14 | 2 · imagined both fists | 3 · imagined both feet |

In [ ]:
# Read one recording and look at its labels
import mne

edf_path = RAW / 'S001' / 'S001R04.edf'
print('File exists:', edf_path.exists())
print('File location:', edf_path)
assert edf_path.exists(), 'Run the ZIP extraction cell above first.'

raw = mne.io.read_raw_edf(str(edf_path), preload=False, verbose='ERROR')
print('Channels:', len(raw.ch_names))
print('Sampling rate:', raw.info['sfreq'])
print('Annotations:', list(raw.annotations.description[:12]))


In [ ]:
# The same T1 means something different in Run 04 vs Run 06
def get_label(run, event):
    if event == 'T0':
        return None
    if run in ['04', '08', '12']:
        return {'T1': 0, 'T2': 1}[event]
    return {'T1': 2, 'T2': 3}[event]

for run in ['04', '06']:
    for event in ['T0', 'T1', 'T2']:
        print('Run', run, event, '→', get_label(run, event))


## 5 · Rebuild the original preprocessed LMDB

**Unchanged from your T4 notebook:** 64 fixed-order channels → average reference → 0.3 Hz high-pass → 60 Hz notch → 200 Hz → 4-second epochs → four 1-second patches. Trial shape is `[64,4,200]`, stored in microvolts and divided by 100 by the loader. This preprocessing runs on **CPU**, so the A100 primarily accelerates fine-tuning, not EDF preprocessing. A subsequent execution in the *same runtime* reuses the local LMDB.

In [ ]:
# Keep exactly the same channel order as the authors
channels = [
    'Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.',
    'C5..', 'C3..', 'C1..', 'Cz..', 'C2..', 'C4..', 'C6..',
    'Cp5.', 'Cp3.', 'Cp1.', 'Cpz.', 'Cp2.', 'Cp4.', 'Cp6.',
    'Fp1.', 'Fpz.', 'Fp2.', 'Af7.', 'Af3.', 'Afz.', 'Af4.', 'Af8.',
    'F7..', 'F5..', 'F3..', 'F1..', 'Fz..', 'F2..', 'F4..',
    'F6..', 'F8..', 'Ft7.', 'Ft8.', 'T7..', 'T8..', 'T9..',
    'T10.', 'Tp7.', 'Tp8.', 'P7..', 'P5..', 'P3..', 'P1..',
    'Pz..', 'P2..', 'P4..', 'P6..', 'P8..', 'Po7.', 'Po3.',
    'Poz.', 'Po4.', 'Po8.', 'O1..', 'Oz..', 'O2..', 'Iz..'
]
print('Channels:', len(channels))
assert len(channels) == 64


In [ ]:
import lmdb, pickle
subjects = [f'S{i:03d}' for i in range(1, 110)]
split_subjects = {
    'train': subjects[:70],    # S001–S070
    'val': subjects[70:89],    # S071–S089
    'test': subjects[89:109]   # S090–S109
}
print({split: (group[0], group[-1], len(group))
       for split, group in split_subjects.items()})

In [ ]:
# SAFE DEFAULT: reuse an already completed local LMDB when rerunning cells.
# Set True only if you explicitly want to delete/rebuild the LOCAL LMDB.
FORCE_REBUILD_LMDB = False

if FORCE_REBUILD_LMDB and DB_PATH.exists():
    shutil.rmtree(DB_PATH)
DB_PATH.mkdir(parents=True, exist_ok=True)

env = lmdb.open(str(DB_PATH), map_size=8 * 1024**3)
with env.begin() as txn:
    previous_keys = txn.get(b'__keys__')
    finished = previous_keys is not None
    existing_entries = txn.stat()['entries']

if finished:
    print('Using the already completed local LMDB (verified in the next cell).')
else:
    if existing_entries > 0:
        env.close()
        raise RuntimeError(
            f'Partial LMDB exists at {DB_PATH}. '
            'Set FORCE_REBUILD_LMDB=True and rerun this cell to rebuild it.'
        )

    all_keys = {'train': [], 'val': [], 'test': []}
    for split, group in split_subjects.items():
        for subject in group:
            for run in runs:
                edf = RAW / subject / f'{subject}R{run}.edf'
                assert edf.is_file(), f'Missing EDF: {edf}. Rerun extraction.'
                raw = mne.io.read_raw_edf(str(edf), preload=True, verbose='ERROR')
                raw.pick_channels(channels, ordered=True)
                if raw.info['bads']:
                    raw.interpolate_bads(verbose='ERROR')
                raw.set_eeg_reference(ref_channels='average', verbose='ERROR')
                raw.filter(l_freq=0.3, h_freq=None, verbose='ERROR')
                raw.notch_filter(60, verbose='ERROR')
                raw.resample(200, verbose='ERROR')

                events, event_ids = mne.events_from_annotations(raw, verbose='ERROR')
                assert set(event_ids) == {'T0', 'T1', 'T2'}, (edf.name, event_ids)
                epochs = mne.Epochs(
                    raw, events, event_id=event_ids,
                    tmin=0, tmax=4 - 1 / 200,
                    baseline=None, preload=True, verbose='ERROR'
                )
                data = epochs.get_data(units='uV')[:, :, -800:]
                data = data.reshape(-1, 64, 4, 200)
                names_by_id = {number: name for name, number in event_ids.items()}

                with env.begin(write=True) as txn:
                    for i, (sample, event_id) in enumerate(zip(data, epochs.events[:, 2])):
                        label = get_label(run, names_by_id[event_id])
                        if label is None:
                            continue
                        key = f'{subject}R{run}-{i}'
                        txn.put(key.encode(), pickle.dumps({'sample': sample, 'label': label}))
                        all_keys[split].append(key)
                del raw, epochs, data

            if int(subject[1:]) % 10 == 0:
                print(split, subject, len(all_keys[split]), 'trials so far', flush=True)

    with env.begin(write=True) as txn:
        txn.put(b'__keys__', pickle.dumps(all_keys))
    print('Finished LMDB preprocessing.')

env.close()

## 6 · Check the dataset **before** training

The original working run produced **9,837 non-rest trials**: **6,300 train + 1,734 validation + 1,803 test**. Stop if these totals change. Trial shapes and labels are checked, too.

In [ ]:
from collections import Counter

env = lmdb.open(str(DB_PATH), readonly=True, lock=False)
with env.begin() as txn:
    key_bytes = txn.get(b'__keys__')
    assert key_bytes is not None, 'LMDB was not completed. Rerun preprocessing.'
    all_keys = pickle.loads(key_bytes)
    for split, keys in all_keys.items():
        labels = [pickle.loads(txn.get(key.encode()))['label'] for key in keys]
        assert set(labels) == {0, 1, 2, 3}, (split, Counter(labels))
        print(split, len(keys), dict(sorted(Counter(labels).items())))
    first = pickle.loads(txn.get(all_keys['train'][0].encode()))
env.close()

expected_counts = {'train': 6300, 'val': 1734, 'test': 1803}
actual_counts = {split: len(keys) for split, keys in all_keys.items()}
assert actual_counts == expected_counts, (actual_counts, expected_counts)
assert first['sample'].shape == (64, 4, 200), first['sample'].shape
assert first['label'] in {0, 1, 2, 3}
print('First trial shape:', first['sample'].shape, '| label:', first['label'])
print('TOTAL:', sum(actual_counts.values()), 'trials — dataset checks passed.')

In [ ]:
# Plot one four-second EEG channel from a labeled trial
import matplotlib.pyplot as plt

trial = first['sample']
print('Class names: 0=left, 1=right, 2=both fists, 3=both feet')
plt.figure(figsize=(10, 3))
plt.plot(np.arange(800) / 200, trial[8].reshape(800))
plt.xlabel('Time (seconds)')
plt.ylabel('Voltage (microvolts)')
plt.title(f"One electrode, one 4-second trial (label = {first['label']})")
plt.grid(alpha=0.2)
plt.show()

## 7 · Download the authors' pretrained CBraMod backbone

The only input on Drive remains the dataset ZIP. The pretrained checkpoint is downloaded **directly to the Colab runtime** from [the authors' Hugging Face repository](https://huggingface.co/weighting666/CBraMod). A SHA-256 check guards against accidentally using different weights.

In [ ]:
from huggingface_hub import hf_hub_download

CHECKPOINT = Path(hf_hub_download(
    repo_id='weighting666/CBraMod',
    filename='pretrained_weights.pth',
    local_dir=str(WORK / 'pretrained_weights')
))
EXPECTED_CHECKPOINT_SHA256 = '0792cb808c14e6b7a2bb2ce1dff379bc47bc54c49a779825bdfeb33bf8157178'
sha = hashlib.sha256()
with CHECKPOINT.open('rb') as f:
    for chunk in iter(lambda: f.read(4 * 1024 * 1024), b''):
        sha.update(chunk)
checkpoint_sha256 = sha.hexdigest()
print('Pretrained checkpoint:', CHECKPOINT)
print('SHA-256:', checkpoint_sha256)
assert checkpoint_sha256 == EXPECTED_CHECKPOINT_SHA256, (
    'Downloaded checkpoint differs from the checked version. '
    'Verify the Hugging Face file before proceeding.'
)

## 8 · Verify the input/output shapes with an A100 forward pass

A single sample is `[1, 64, 4, 200]`; pretrained CBraMod returns patch features `[1, 64, 4, 200]`; the new task head returns four logits `[1, 4]`. These are **not yet trained predictions**, since the four-class head is randomly initialized.

In [ ]:
import sys
from types import SimpleNamespace

if str(CBRAMOD) not in sys.path:
    sys.path.insert(0, str(CBRAMOD))

from models.model_for_physio import Model

params = SimpleNamespace(
    cuda=0, use_pretrained_weights=True, foundation_dir=str(CHECKPOINT),
    classifier='all_patch_reps', num_of_classes=4, dropout=0.1
)

model = Model(params).cuda().eval()
x = torch.from_numpy(first['sample'] / 100).float().unsqueeze(0).cuda()

with torch.no_grad():
    features = model.backbone(x)
    logits = model(x)

print('Input:', tuple(x.shape))
print('Transformer features:', tuple(features.shape))
print('Final logits:', tuple(logits.shape))
print('Probability of each class:', logits.softmax(-1).cpu().numpy().round(3))
del model, features, logits, x
torch.cuda.empty_cache()


## 9 · A100 optimization layer (applied only in the local runtime)

The upstream code is pinned to a known commit; the small patch below makes its **PhysioNet LMDB loader** share one connection and optionally cache all trial tensors in CPU RAM (~2.0 GiB). Data order, scaling, labels and model stay the same. It also turns on pinned-memory data loading. In fast mode the published trainer/evaluator use **BF16 autocast** for forward passes, **TF32** for float32 matmul, and avoid a costly loss-to-CPU synchronization every step. AdamW, cosine schedule, gradient clipping, 50 epochs and best-validation-kappa selection remain unchanged.

`FAST_A100 = False` disables BF16/TF32, RAM cache and worker processes for a closer FP32 reference. Optimizations change floating-point and/or loader execution details; treat fast mode as a speed-optimized reproduction, not a bitwise replication.

In [ ]:
# Idempotent, version-specific local performance patch; NEVER alters your GitHub repository.
# Checks every exact upstream snippet before modifying, failing loudly if source differs.
def patch_exact(path, replacements):
    source = path.read_text()
    changes = 0
    for name, before, after in replacements:
        if after in source:
            print(f'{path.name}: {name} already applied')
            continue
        matches = source.count(before)
        if matches != 1:
            raise RuntimeError(f'{path}: expected ONE match for {name}, got {matches}. Source may have changed.')
        source = source.replace(before, after, 1)
        changes += 1
    compile(source, str(path), 'exec')
    if changes:
        path.write_text(source)
    print(path.name, 'new patches:', changes)

loader_path = CBRAMOD / 'datasets' / 'physio_dataset.py'
loader_init_old = """        self.db = lmdb.open(data_dir, readonly=True, lock=False, readahead=True, meminit=False)
        with self.db.begin(write=False) as txn:
            self.keys = pickle.loads(txn.get('__keys__'.encode()))[mode]"""
loader_init_new = """        if not hasattr(CustomDataset, '_shared_db'):
            CustomDataset._shared_db = lmdb.open(data_dir, readonly=True, lock=False, readahead=True, meminit=False)
        self.db = CustomDataset._shared_db
        with self.db.begin(write=False) as txn:
            self.keys = pickle.loads(txn.get('__keys__'.encode()))[mode]
        # Optional one-time RAM cache avoids deserializing ~2 GiB every epoch.
        if os.environ.get('CBRAMOD_RAM_CACHE', '0') == '1':
            self.ram_x = np.empty((len(self.keys), 64, 4, 200), dtype=np.float32)
            self.ram_y = np.empty(len(self.keys), dtype=np.int64)
            with self.db.begin(write=False) as txn:
                for idx, key in enumerate(self.keys):
                    sample = pickle.loads(txn.get(key.encode()))
                    self.ram_x[idx] = (sample['sample'] / 100).astype(np.float32)
                    self.ram_y[idx] = sample['label']
            print(f'Loaded {mode} into RAM: {len(self.keys)} trials')"""
item_old = """    def __getitem__(self, idx):
        key = self.keys[idx]"""
item_new = """    def __getitem__(self, idx):
        if hasattr(self, 'ram_x'):
            return self.ram_x[idx], int(self.ram_y[idx])
        key = self.keys[idx]"""
loader_old_tr = """                shuffle=True,
            ),"""
loader_new_tr = """                shuffle=True,
                num_workers=self.params.num_workers,
                pin_memory=True,
                persistent_workers=self.params.num_workers > 0,
            ),"""
# Distinguish validation from test by replacing the two loader blocks as a single string.
loader_old_eval = """                shuffle=False,
            ),
            'test': DataLoader("""
loader_new_eval = """                shuffle=False,
                num_workers=self.params.num_workers,
                pin_memory=True,
                persistent_workers=self.params.num_workers > 0,
            ),
            'test': DataLoader("""
loader_old_test = """                shuffle=False,
            ),
        }"""
loader_new_test = """                shuffle=False,
                num_workers=self.params.num_workers,
                pin_memory=True,
                persistent_workers=self.params.num_workers > 0,
            ),
        }"""
patch_exact(loader_path, [
    ('shared LMDB + optional RAM cache', loader_init_old, loader_init_new),
    ('cached __getitem__', item_old, item_new),
    ('train workers', loader_old_tr, loader_new_tr),
    ('validation workers', loader_old_eval, loader_new_eval),
    ('test workers', loader_old_test, loader_new_test),
])

trainer_path = CBRAMOD / 'finetune_trainer.py'
trainer_before = """                self.optimizer.zero_grad()
                x = x.cuda()
                y = y.cuda()
                pred = self.model(x)
                if self.params.downstream_dataset == 'ISRUC':
                    loss = self.criterion(pred.transpose(1, 2), y)
                else:
                    loss = self.criterion(pred, y)
                loss.backward()
                losses.append(loss.data.cpu().numpy())"""
trainer_after = """                self.optimizer.zero_grad(set_to_none=True)
                x = x.cuda(non_blocking=True)
                y = y.cuda(non_blocking=True)
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16,
                                    enabled=os.environ.get('CBRAMOD_AMP_MODE') == 'bf16'):
                    pred = self.model(x)
                    if self.params.downstream_dataset == 'ISRUC':
                        loss = self.criterion(pred.transpose(1, 2), y)
                    else:
                        loss = self.criterion(pred, y)
                loss.backward()
                losses.append(loss.detach())"""
trainer_mean_before = "                        np.mean(losses),\n                        acc,\n                        kappa,"
trainer_mean_after = "                        torch.stack(losses).mean().item(),\n                        acc,\n                        kappa,"
patch_exact(trainer_path, [
    ('BF16 / asynchronous transfers / no per-step sync', trainer_before, trainer_after),
    ('aggregate training loss once per epoch', trainer_mean_before, trainer_mean_after),
])

evaluator_path = CBRAMOD / 'finetune_evaluator.py'
patch_exact(evaluator_path, [
    ('import env settings', 'import numpy as np\nimport torch', 'import numpy as np\nimport os\nimport torch'),
    ('BF16 validation and test', """            x = x.cuda()
            y = y.cuda()

            pred = model(x)
            pred_y = torch.max(pred, dim=-1)[1]""",
     """            x = x.cuda(non_blocking=True)
            y = y.cuda(non_blocking=True)

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16,
                                enabled=os.environ.get('CBRAMOD_AMP_MODE') == 'bf16'):
                pred = model(x)
            pred_y = torch.max(pred, dim=-1)[1]"""),
])

main_path = CBRAMOD / 'finetune_main.py'
patch_exact(main_path, [
    ('TF32 toggle', '    print(params)\n    setup_seed(params.seed)',
     """    print(params)
    # Toggle faster A100 tensor-core math; leave disabled in reference-FP32 mode.
    tf32 = os.environ.get('CBRAMOD_TF32') == '1'
    torch.backends.cuda.matmul.allow_tf32 = tf32
    torch.backends.cudnn.allow_tf32 = tf32
    setup_seed(params.seed)"""),
    ('import env', 'import argparse\nimport random', 'import argparse\nimport os\nimport random'),
])
print('Local A100 optimization patch passed Python syntax checks.')

In [ ]:
# Check the official loader after patching without doing the expensive full RAM warmup.
from types import SimpleNamespace
from datasets.physio_dataset import LoadDataset
prior_cache_flag = os.environ.get('CBRAMOD_RAM_CACHE')
os.environ['CBRAMOD_RAM_CACHE'] = '0'
try:
    loaded = LoadDataset(SimpleNamespace(
        datasets_dir=str(DB_PATH), batch_size=BATCH_SIZE, num_workers=0
    )).get_data_loader()
    xb, yb = next(iter(loaded['train']))
    print('Training batch:', tuple(xb.shape), '| labels:', tuple(yb.shape), '| dtype:', xb.dtype)
    assert xb.shape == (BATCH_SIZE, 64, 4, 200)
    assert yb.shape == (BATCH_SIZE,)
    assert xb.dtype == torch.float32
    assert set(loaded) == {'train', 'val', 'test'}
    del loaded, xb, yb
finally:
    if prior_cache_flag is None:
        os.environ.pop('CBRAMOD_RAM_CACHE', None)
    else:
        os.environ['CBRAMOD_RAM_CACHE'] = prior_cache_flag

## 10 · Fine-tune (accelerated A100 / reference FP32)

The normal run remains **50 epochs, batch 64, seed 3407**, with trainable CBraMod backbone, AdamW, cosine LR schedule, dropout 0.1, label smoothing 0.1 and gradient clipping. Select **fast mode** to focus on wall-clock speed, or **reference FP32** to get closer to T4 arithmetic. A100 speedups depend on data-loading overhead, CPU and runtime configuration, so no fixed time reduction is promised.

Run artifacts are first written locally; each completed seed is copied under `results/physionet_mi_4class/<experiment>/runs/seed_XXXX/`. Completed runs are skipped on reruns unless `FORCE_RERUN=True`. To run a new independent version, change `EXPERIMENT_SUFFIX` at the top.

In [ ]:
# Original published hyperparameters; speed optimizations are controlled separately.
BACKBONE_LR = 0.0001
WEIGHT_DECAY = 0.05
DROPOUT = 0.1
LABEL_SMOOTHING = 0.1
CLIP_VALUE = 1

RUNS.mkdir(parents=True, exist_ok=True)
DRIVE_EXPERIMENT.mkdir(parents=True, exist_ok=True)
(DRIVE_EXPERIMENT / 'metrics').mkdir(exist_ok=True)
RUN_CONFIG = {
    'experiment_id': EXPERIMENT_ID,
    'mode': MODE,
    'fast_a100': FAST_A100,
    'autocast': 'bfloat16' if FAST_A100 else 'disabled',
    'tf32': FAST_A100,
    'ram_cache': FAST_A100,
    'num_workers': NUM_WORKERS,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'seeds': SEEDS,
    'backbone_lr': BACKBONE_LR,
    'weight_decay': WEIGHT_DECAY,
    'dropout': DROPOUT,
    'label_smoothing': LABEL_SMOOTHING,
    'clip_value': CLIP_VALUE,
    'classifier': 'all_patch_reps',
    'class_count': 4,
}
print(json.dumps(RUN_CONFIG, indent=2))
print('Output:', DRIVE_EXPERIMENT)

In [ ]:
import csv
from datetime import datetime, timezone

for seed in SEEDS:
    run_name = f'seed_{seed}'
    run_dir = RUNS / run_name
    target_dir = DRIVE_EXPERIMENT / 'runs' / run_name
    model_dir = run_dir / 'checkpoints'
    log_dir = run_dir / 'logs'
    metrics_dir = run_dir / 'metrics'
    for p in [model_dir, log_dir, metrics_dir]:
        p.mkdir(parents=True, exist_ok=True)

    if (target_dir / 'completed.json').is_file() and not FORCE_RERUN:
        print(f'{run_name}: complete in Drive; skipping. Change EXPERIMENT_SUFFIX for a separate run.')
        continue

    if FORCE_RERUN and target_dir.is_dir():
        shutil.rmtree(target_dir)
    if FORCE_RERUN and run_dir.is_dir():
        shutil.rmtree(run_dir)
        for p in [model_dir, log_dir, metrics_dir]:
            p.mkdir(parents=True, exist_ok=True)

    # A new attempt must not be confused with a checkpoint from a failed attempt.
    for stale_checkpoint in model_dir.glob('*.pth'):
        stale_checkpoint.unlink()

    run_config = {**RUN_CONFIG, 'seed': seed, 'source_commit': commit, 'checkpoint_sha256': checkpoint_sha256}
    (run_dir / 'config.json').write_text(json.dumps(run_config, indent=2) + '\n')

    command = [
        sys.executable, '-u', 'finetune_main.py',
        '--downstream_dataset', 'PhysioNet-MI',
        '--datasets_dir', str(DB_PATH),
        '--num_of_classes', '4',
        '--classifier', 'all_patch_reps',
        '--foundation_dir', str(CHECKPOINT),
        '--model_dir', str(model_dir),
        '--cuda', '0', '--seed', str(seed),
        '--epochs', str(EPOCHS), '--batch_size', str(BATCH_SIZE),
        '--lr', str(BACKBONE_LR), '--weight_decay', str(WEIGHT_DECAY),
        '--dropout', str(DROPOUT), '--label_smoothing', str(LABEL_SMOOTHING),
        '--clip_value', str(CLIP_VALUE),
        '--num_workers', str(NUM_WORKERS),
    ]
    env_vars = os.environ.copy()
    env_vars.update({
        'CBRAMOD_AMP_MODE': 'bf16' if FAST_A100 else 'fp32',
        'CBRAMOD_TF32': '1' if FAST_A100 else '0',
        'CBRAMOD_RAM_CACHE': '1' if FAST_A100 else '0',
        'PYTHONUNBUFFERED': '1',
    })
    log_path = log_dir / 'train.log'
    print(f'\nTraining {run_name} on {gpu_name} ({MODE}).')
    started = time.monotonic()
    with log_path.open('w', encoding='utf-8') as logfile:
        process = subprocess.Popen(command, cwd=str(CBRAMOD), env=env_vars,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            logfile.write(line)
            logfile.flush()
        return_code = process.wait()

    elapsed_minutes = (time.monotonic() - started) / 60
    if return_code:
        print('Training failed. Copying the log to Drive for debugging.')
        target_dir.mkdir(parents=True, exist_ok=True)
        shutil.copytree(log_dir, target_dir / 'logs', dirs_exist_ok=True)
        raise RuntimeError(f'{run_name} failed (exit code {return_code}). See {target_dir}/logs/train.log')

    # Official trainer saves only the best-validation-kappa weights after test evaluation.
    checkpoints = list(model_dir.glob('*.pth'))
    if len(checkpoints) != 1:
        raise RuntimeError(f'Expected one best model checkpoint for {run_name}; found {checkpoints}')
    checkpoints[0].rename(model_dir / 'best_model.pth')

    text_log = log_path.read_text(encoding='utf-8')
    epoch_pattern = (r'Epoch\s+(\d+)\s+:\s+Training Loss:\s+([0-9.eE+-]+),\s+'
                     r'acc:\s+([0-9.eE+-]+),\s+kappa:\s+([0-9.eE+-]+),\s+'
                     r'f1:\s+([0-9.eE+-]+),\s+LR:\s+([0-9.eE+-]+),\s+'
                     r'Time elapsed\s+([0-9.eE+-]+)\s+mins')
    epochs_found = re.findall(epoch_pattern, text_log)
    epoch_rows = [dict(zip(['epoch', 'training_loss', 'val_balanced_accuracy', 'val_kappa',
                            'val_weighted_f1', 'backbone_lr', 'epoch_minutes'],
                           [int(v[0])] + [float(t) for t in v[1:]])) for v in epochs_found]
    if len(epoch_rows) != EPOCHS:
        raise RuntimeError(f'Expected {EPOCHS} recorded epochs, got {len(epoch_rows)}. Inspect {log_path}')
    with (metrics_dir / 'epoch_metrics.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(epoch_rows[0])); writer.writeheader(); writer.writerows(epoch_rows)

    match = re.findall(r'Test Evaluation: acc:\s*([0-9.]+), kappa:\s*([0-9.]+), f1:\s*([0-9.]+)', text_log)
    if not match:
        raise RuntimeError(f'Test results missing in {log_path}')
    a, k, f1 = map(float, match[-1])
    test_metrics = {'seed': seed, 'balanced_accuracy': a, 'cohen_kappa': k, 'weighted_f1': f1,
                    'elapsed_minutes': round(elapsed_minutes, 2)}
    (metrics_dir / 'test_metrics.json').write_text(json.dumps(test_metrics, indent=2) + '\n')
    (run_dir / 'completed.json').write_text(json.dumps({
        'completed_utc': datetime.now(timezone.utc).isoformat(),
        'seed': seed, 'experiment_id': EXPERIMENT_ID,
        'checkpoint': 'checkpoints/best_model.pth',
        'test_metrics': test_metrics,
    }, indent=2) + '\n')
    # Copy completion marker LAST so a partially copied Drive run is never mistaken for complete.
    (target_dir / 'completed.json').unlink(missing_ok=True)
    shutil.copytree(run_dir, target_dir, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('completed.json'))
    shutil.copy2(run_dir / 'completed.json', target_dir / 'completed.json')
    print(f'Saved {run_name}: {target_dir}')
    print('Test:', test_metrics)

## 11 · Consolidated test scores and learning curves

Reads completed run files **from Google Drive**, not the ephemeral training directory, so you can reopen the notebook to inspect results later. `acc` in the authors' log is **balanced accuracy**. Separate GPU-optimized results from the earlier T4 FP32 run; a faster GPU does not imply higher model accuracy.

In [ ]:
import pandas as pd
rows = []
all_epochs = []
for seed in SEEDS:
    run_dir = DRIVE_EXPERIMENT / 'runs' / f'seed_{seed}'
    score_path = run_dir / 'metrics' / 'test_metrics.json'
    epochs_path = run_dir / 'metrics' / 'epoch_metrics.csv'
    if score_path.is_file():
        rows.append(json.loads(score_path.read_text()))
    else:
        print('Not completed:', seed)
    if epochs_path.is_file():
        frame = pd.read_csv(epochs_path)
        frame.insert(0, 'seed', seed)
        all_epochs.append(frame)

results = pd.DataFrame(rows)
if len(results):
    display(results)
    print('\nMean test scores:')
    print(results[['balanced_accuracy', 'cohen_kappa', 'weighted_f1']].mean().round(5))
    if len(results) > 1:
        print('\nAcross-seed standard deviation:')
        print(results[['balanced_accuracy', 'cohen_kappa', 'weighted_f1']].std().round(5))
print('\nPrevious working T4 / seed 3407: balanced accuracy 0.63457, kappa 0.51269, F1 0.63527')
print('Paper reference:               balanced accuracy 0.64170, kappa 0.52220, F1 0.64270')

epoch_results = pd.concat(all_epochs, ignore_index=True) if all_epochs else pd.DataFrame()
if len(epoch_results):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 4))
    for seed, group in epoch_results.groupby('seed'):
        ax.plot(group['epoch'], group['val_kappa'], label=f'seed {seed}')
    ax.set(xlabel='Epoch', ylabel='Validation Cohen kappa', title='Validation performance')
    ax.legend(); ax.grid(alpha=0.25)
    plt.show()

## 12 · Experiment metadata and summary files

One experiment folder, one seed folder per training run, no scattered `cbramod_results` directories. These are the stable files to use in later analysis: `metrics/test_scores.csv`, `metrics/epoch_metrics.csv`, `manifest.json`, `environment.json`, and the per-seed checkpoints/logs. The dataset ZIP stays at the Drive root; temporary preprocessed EDF and LMDB data do not consume Drive space.

In [ ]:
DRIVE_EXPERIMENT.mkdir(parents=True, exist_ok=True)
metrics_root = DRIVE_EXPERIMENT / 'metrics'
metrics_root.mkdir(parents=True, exist_ok=True)
if len(results):
    results.to_csv(metrics_root / 'test_scores.csv', index=False)
if len(epoch_results):
    epoch_results.to_csv(metrics_root / 'epoch_metrics.csv', index=False)

manifest = {
    **RUN_CONFIG,
    'dataset_zip': str(ZIP_PATH),
    'dataset_zip_size_bytes': ZIP_PATH.stat().st_size,
    'dataset_trial_counts': {split: len(keys) for split, keys in all_keys.items()},
    'dataset_split_subjects': {split: [group[0], group[-1]] for split, group in split_subjects.items()},
    'class_names': {'0': 'left fist', '1': 'right fist', '2': 'both fists', '3': 'both feet'},
    'imagery_runs': runs,
    'trial_shape': [64, 4, 200],
    'preprocessing': 'average reference / 0.3 Hz HP / 60 Hz notch / 200 Hz / 4 seconds',
    'normalization': 'microvolts / 100',
    'source_commit': commit,
    'pretrained_checkpoint_sha256': checkpoint_sha256,
    'model_selection': 'max validation Cohen kappa',
    'official_trainer': True,
    'local_runtime_performance_patches': [
        'PhysioNet LMDB shared connection and optional RAM cache',
        'pin memory and optional dataloader workers',
        'BF16 autocast (fast mode only)',
        'TF32 float32 matmul (fast mode only)',
        'one synchronization per epoch for training loss',
    ],
}
(DRIVE_EXPERIMENT / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
environment = {'gpu': gpu_name, 'gpu_memory_gib': round(gpu_mem_gb, 2),
               'torch_cuda': str(torch.version.cuda), 'versions': VERSIONS}
(DRIVE_EXPERIMENT / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')

print('\nOrganized Drive directory:', DRIVE_EXPERIMENT)
for path in sorted(DRIVE_EXPERIMENT.rglob('*')):
    if path.is_file():
        print(' -', path.relative_to(DRIVE_EXPERIMENT),
              f'({path.stat().st_size/1024**2:.1f} MiB)')

## Final directory layout and reproducibility notes

On first execution, `CBraMod-MI` only needs the ZIP. The notebook then creates the following general-purpose results structure:

```text
MyDrive/
└── CBraMod-MI/
    ├── eeg-motor-movementimagery-dataset-1.0.0.zip  # your existing input
    └── results/
        └── physionet_mi_4class/
            └── a100_fast_bf16_bs64_e50/             # mode + training settings
                ├── manifest.json                     # configuration and provenance
                ├── environment.json                  # GPU and software versions
                ├── metrics/
                │   ├── test_scores.csv               # across-seed summary
                │   └── epoch_metrics.csv             # training curves
                └── runs/
                    └── seed_3407/
                        ├── config.json
                        ├── completed.json
                        ├── checkpoints/
                        │   └── best_model.pth
                        ├── logs/
                        │   └── train.log
                        └── metrics/
                            ├── epoch_metrics.csv
                            └── test_metrics.json
```

**Behavior:** Only completed seed folders are regarded as successful. Completed seeds are skipped by default, preventing accidental overwritten results. For a *fresh independent repetition*, set a unique `EXPERIMENT_SUFFIX` in the first code cell (e.g. `replicate_02`). For a full FP32 reference run, set `FAST_A100=False` (it uses a separate experiment ID). Changing seed(s) also keeps each run in a separate folder.

**Speed notes:** An A100 is expected to accelerate training, especially with BF16, but preprocessing is mostly CPU-bound and must be performed once on first session. The optional RAM cache trades ~2 GiB of Colab **system RAM** for avoiding repeated LMDB decoding. Metrics can differ from T4 due to BF16/TF32, environment versions, and GPU behavior. Do not treat the accelerated scores as an exact FP32 replay.

Official references: [CBraMod trainer](https://github.com/wjq-learning/CBraMod/blob/main/finetune_trainer.py) · [PhysioNet dataset loader](https://github.com/wjq-learning/CBraMod/blob/main/datasets/physio_dataset.py) · [paper](https://openreview.net/forum?id=NPNUHgHF2w).